# Grade 11 flagship — Climate or weather?

> **SYNTHETIC** table. Use PPDAC headings. After `python tools/fetch_data.py climate-monthly` (licence: CC BY 4.0, credit World Bank CCKP) the same code runs on real data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

clim = pd.read_csv(data_file("climate-monthly", "climate_monthly_synthetic.csv"))
print(clim.shape)
import sys
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)   # record your environment
SEED = 2026

## P — Problem and Plan
*Question, definition of 'trend', what would change your mind?*

## D — Data
*Source, licence, cleaning log.*

## A — Analysis

In [ ]:
clim["t"] = clim["year"] + (clim["month"] - 1) / 12
seasonal = clim.groupby("month")["mean_temp_c"].mean() - clim["mean_temp_c"].mean()
clim["deseason"] = clim["mean_temp_c"] - clim["month"].map(seasonal)
slope = 10 * np.polyfit(clim["t"], clim["deseason"], 1)[0]

rng = np.random.default_rng(SEED)
years = clim["year"].unique()
boot = []
for _ in range(1000):
    pick = rng.choice(years, len(years), replace=True)
    d = pd.concat([clim[clim["year"] == y] for y in pick])
    boot.append(10 * np.polyfit(d["t"], d["deseason"], 1)[0])
lo, hi = np.percentile(boot, [2.5, 97.5])
assert lo < slope < hi
print(f"trend {slope:.2f} °C/decade, 95% interval {lo:.2f} to {hi:.2f}")

## C — Conclusion
*Claim with interval · Reasoning · Limits (synthetic? one series? only temperature?) · Reproduce: environment printed above, seed = 2026 · AI help used for: ___*